# Phase 2 — Shopee Auto-Labeling with Best Fine-Tuned Joint DeBERTa


In [ ]:
# ============================================================
# 1. Optional package check / install
# ============================================================

# !pip install -q transformers accelerate sentencepiece pandas numpy scikit-learn tqdm openpyxl

import os
import gc
import json
import math
import random
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

pd.set_option("display.max_colwidth", 180)
tqdm.pandas()

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.12.1+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
# ============================================================
# 2. Path configuration
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\Lenovo\Documents\Skripsi")

# Input from preprocessing:
INPUT_CSV_PATH = PROJECT_ROOT / "dataset" / "phase2" / "shopee" / "05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv"

# Fine-tuning output folder from:
# fine_tune_deberta_base_joint_aspect_sentiment_colab_stage4_clean.ipynb
PHASE1_OUTPUT_DIR = (
    PROJECT_ROOT
    / "dataset"
    / "phase1"
    / "trustpilot_reformatted_output"
    / "deberta_joint_aspect_sentiment_output_colab_base"
)

BEST_MODEL_DIR = PHASE1_OUTPUT_DIR / "colab_best_joint_deberta_model"
BEST_THRESHOLD_PATH = PHASE1_OUTPUT_DIR / "colab_best_per_label_thresholds.json"

# Output folder for Shopee auto-labeling result.
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "phase2" / "shopee" / "autolabel_deberta_14aspect"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fallbacks for Colab / uploaded-file testing.
fallback_input_paths = [
    INPUT_CSV_PATH,
    Path("/content/05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv"),
    Path("05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv"),
]

for p in fallback_input_paths:
    if p.exists():
        INPUT_CSV_PATH = p
        break

fallback_model_dirs = [
    BEST_MODEL_DIR,
    Path("/content/drive/MyDrive/Skripsi/dataset/phase1/trustpilot_reformatted_output/deberta_joint_aspect_sentiment_output_colab_base/colab_best_joint_deberta_model"),
    Path("/content/colab_best_joint_deberta_model"),
    Path("colab_best_joint_deberta_model"),
]

for p in fallback_model_dirs:
    if p.exists():
        BEST_MODEL_DIR = p
        PHASE1_OUTPUT_DIR = BEST_MODEL_DIR.parent
        break

fallback_threshold_paths = [
    BEST_THRESHOLD_PATH,
    PHASE1_OUTPUT_DIR / "colab_best_per_label_thresholds.json",
    Path("/content/drive/MyDrive/Skripsi/dataset/phase1/trustpilot_reformatted_output/deberta_joint_aspect_sentiment_output_colab_base/colab_best_per_label_thresholds.json"),
    Path("/content/colab_best_per_label_thresholds.json"),
    Path("colab_best_per_label_thresholds.json"),
]

for p in fallback_threshold_paths:
    if p.exists():
        BEST_THRESHOLD_PATH = p
        break

print("INPUT_CSV_PATH      :", INPUT_CSV_PATH)
print("BEST_MODEL_DIR      :", BEST_MODEL_DIR)
print("BEST_THRESHOLD_PATH :", BEST_THRESHOLD_PATH)
print("OUTPUT_DIR          :", OUTPUT_DIR)

if not INPUT_CSV_PATH.exists():
    raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV_PATH}")

if not BEST_MODEL_DIR.exists():
    raise FileNotFoundError(
        "Best model folder not found. Make sure the fine-tuning notebook has produced/copy-saved the best model to:\\n"
        f"{BEST_MODEL_DIR}"
    )

if not BEST_THRESHOLD_PATH.exists():
    raise FileNotFoundError(
        "Best threshold JSON not found. Make sure the fine-tuning notebook created:\\n"
        f"{BEST_THRESHOLD_PATH}"
    )

INPUT_CSV_PATH      : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv
BEST_MODEL_DIR      : C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base\colab_best_joint_deberta_model
BEST_THRESHOLD_PATH : C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base\colab_best_per_label_thresholds.json
OUTPUT_DIR          : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect


In [ ]:
# ============================================================
# 3. Label schema — must match the fine-tuned Trustpilot model
# ============================================================

ASPECT_COLS = [
    "Information",
    "Product_features",
    "Item_quality",
    "App_experience",
    "Pricing",
    "Product_availability",
    "Payment",
    "Delivered_product_status",
    "Shipping",
    "Return_process",
    "Refund_process",
    "Customer_service",
    "Trust",
]

JOINT_LABEL_COLS = []
for aspect in ASPECT_COLS:
    JOINT_LABEL_COLS.extend([
        f"{aspect}_Positive",
        f"{aspect}_Negative",
    ])

LABEL_TO_ID = {label: i for i, label in enumerate(JOINT_LABEL_COLS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

print("Number of aspects:", len(ASPECT_COLS))
print("Number of joint labels:", len(JOINT_LABEL_COLS))
print(JOINT_LABEL_COLS)

Number of aspects: 14
Number of joint labels: 28
['Information_Positive', 'Information_Negative', 'Product_features_Positive', 'Product_features_Negative', 'Item_quality_Positive', 'Item_quality_Negative', 'App_experience_Positive', 'App_experience_Negative', 'Pricing_Positive', 'Pricing_Negative', 'Product_availability_Positive', 'Product_availability_Negative', 'Payment_Positive', 'Payment_Negative', 'Packaging_Positive', 'Packaging_Negative', 'Delivered_product_status_Positive', 'Delivered_product_status_Negative', 'Shipping_Positive', 'Shipping_Negative', 'Return_process_Positive', 'Return_process_Negative', 'Refund_process_Positive', 'Refund_process_Negative', 'Customer_service_Positive', 'Customer_service_Negative', 'Trust_Positive', 'Trust_Negative']


In [ ]:
# ============================================================
# 4. Inference configuration
# ============================================================
# Best run was:
# colab_small_lr3e-5_len256_unweighted_epoch8
# use MAX_LENGTH = 256.

SELECTED_BEST_RUN_NAME = "colab_small_lr3e-5_len256_unweighted_epoch8"

TEXT_COL = "english_review"
ID_COL = "id"

MAX_LENGTH = 256
BATCH_SIZE = 8  # safe for RTX 3050 4GB and Colab T4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# limit rows for a quick test. Use None for full auto-labeling.
SMOKE_TEST_ROWS = None  # e.g. 100

# Output files.
AUTO_FULL_PATH = OUTPUT_DIR / "06_shopee_autolabeled_deberta_14aspect_full_probabilities.csv"
AUTO_COMPACT_PATH = OUTPUT_DIR / "06_shopee_autolabeled_deberta_14aspect_compact.csv"
AUTO_PASS_ONLY_PATH = OUTPUT_DIR / "06_shopee_autolabeled_deberta_14aspect_pass_only.csv"
NO_ASPECT_AUDIT_PATH = OUTPUT_DIR / "06_shopee_no_14aspect_predicted_audit.csv"
HUMAN_VERIFICATION_TEMPLATE_PATH = OUTPUT_DIR / "07_shopee_human_verification_template.csv"
RUN_SUMMARY_PATH = OUTPUT_DIR / "06_shopee_autolabeling_run_summary.json"

print("Selected best run:", SELECTED_BEST_RUN_NAME)
print("Device:", DEVICE)
print("MAX_LENGTH:", MAX_LENGTH)
print("BATCH_SIZE:", BATCH_SIZE)

Selected best run: colab_small_lr3e-5_len256_unweighted_epoch8
Device: cuda
MAX_LENGTH: 256
BATCH_SIZE: 8


In [5]:
# ============================================================
# 5. Load Shopee preprocessed dataset
# ============================================================

df = pd.read_csv(INPUT_CSV_PATH)

required_cols = [ID_COL, TEXT_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Found columns: {df.columns.tolist()}")

df = df[required_cols].copy()
df[ID_COL] = df[ID_COL].astype(str)
df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

# Basic safety cleanup.
df = df[df[TEXT_COL].str.len() > 0].copy()
df = df.drop_duplicates(subset=[ID_COL], keep="first").copy()
df = df.drop_duplicates(subset=[TEXT_COL], keep="first").copy()
df = df.reset_index(drop=True)

if SMOKE_TEST_ROWS is not None:
    df = df.head(int(SMOKE_TEST_ROWS)).copy()

print("Rows for auto-labeling:", len(df))
display(df.head(10))

Rows for auto-labeling: 8179


,id,english_review
0,2d68668cf8c3d9b8,i hate your ads
1,d363df62267eae48,"Shopee please do not raise the service fee, from 1,000 then 2,000 now 2,200. If you could please go back to 1,000."
2,8bdfe8eba767c20d,The most maximal apps have pop ads everywhere.
3,c6e103e312de60a4,"Open the application, and then there's this live sale. It's not fun, especially with the full volume, set from the suit, just couldn't help. Please help me solve my problem."
4,c49004b4a142159c,"So satisfied and busy shopping in a diphoope, may the first-rate shooter"
5,52e7ed4683508031,"There's also broken stuff, and something doesn't match the order."
6,45cc42de4e2ba534,"Please don't do too much commercial"" or button passes zoomed in, I'm working on an apk editor show up an ad, and then I press through, and then I go through the apk shopee, and..."
7,83cdbca62c7d8b77,"Too many ads. It's annoying, a little bit of an udh ad on the dial x. Keep it open."
8,bb2df138ba17108d,My paansi is on any application if the shoope trs ad appears
9,cd12de26fc793564,A very helpful application of both easy and safe shopping


In [6]:
# ============================================================
# 6. Load best model and best per-label thresholds
# ============================================================

with open(BEST_THRESHOLD_PATH, "r", encoding="utf-8") as f:
    best_thresholds = json.load(f)

# Validate thresholds.
missing_threshold_labels = [label for label in JOINT_LABEL_COLS if label not in best_thresholds]
extra_threshold_labels = [label for label in best_thresholds.keys() if label not in JOINT_LABEL_COLS]

if missing_threshold_labels:
    raise ValueError(f"Threshold file is missing labels: {missing_threshold_labels}")

if extra_threshold_labels:
    print("Warning: threshold file has extra labels not used by this notebook:")
    print(extra_threshold_labels)

tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))
model = AutoModelForSequenceClassification.from_pretrained(str(BEST_MODEL_DIR))
model.to(DEVICE)
model.float()
model.eval()

print("Loaded tokenizer and model from:", BEST_MODEL_DIR)
print("Loaded thresholds from:", BEST_THRESHOLD_PATH)
print("Model num_labels:", model.config.num_labels)
print("Expected num_labels:", len(JOINT_LABEL_COLS))

if int(model.config.num_labels) != len(JOINT_LABEL_COLS):
    raise ValueError(
        f"Model label count mismatch. Model has {model.config.num_labels}, expected {len(JOINT_LABEL_COLS)}."
    )

# Show threshold summary.
threshold_values = np.array([float(best_thresholds[label]) for label in JOINT_LABEL_COLS], dtype=np.float32)
print("Threshold min :", float(threshold_values.min()))
print("Threshold mean:", float(threshold_values.mean()))
print("Threshold max :", float(threshold_values.max()))

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loaded tokenizer and model from: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base\colab_best_joint_deberta_model
Loaded thresholds from: C:\Users\Lenovo\Documents\Skripsi\dataset\phase1\trustpilot_reformatted_output\deberta_joint_aspect_sentiment_output_colab_base\colab_best_per_label_thresholds.json
Model num_labels: 28
Expected num_labels: 28
Threshold min : 0.10000000149011612
Threshold mean: 0.31785717606544495
Threshold max : 0.75


In [7]:
# ============================================================
# 7. Prediction helper functions
# ============================================================

def clean_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def make_threshold_vector(thresholds, label_cols):
    if isinstance(thresholds, dict):
        return np.array([float(thresholds[label]) for label in label_cols], dtype=np.float32)

    threshold_array = np.asarray(thresholds, dtype=np.float32)

    if threshold_array.ndim == 0:
        threshold_array = np.full(len(label_cols), float(threshold_array), dtype=np.float32)

    if len(threshold_array) != len(label_cols):
        raise ValueError(f"Threshold length mismatch. Expected {len(label_cols)}, got {len(threshold_array)}.")

    return threshold_array


def apply_thresholds_with_conflict_resolution(probs, thresholds, label_cols, aspect_cols):
    """
    Convert probabilities into 0/1 labels using per-label thresholds.

    Conflict rule:
    If the same aspect gets both Positive and Negative, keep only the higher-probability label.
    """
    threshold_vector = make_threshold_vector(thresholds, label_cols)
    preds = (probs >= threshold_vector.reshape(1, -1)).astype(int)

    label_to_idx = {label: i for i, label in enumerate(label_cols)}
    conflict_count = 0

    for row_idx in range(preds.shape[0]):
        for aspect in aspect_cols:
            pos_label = f"{aspect}_Positive"
            neg_label = f"{aspect}_Negative"

            pos_idx = label_to_idx[pos_label]
            neg_idx = label_to_idx[neg_label]

            if preds[row_idx, pos_idx] == 1 and preds[row_idx, neg_idx] == 1:
                conflict_count += 1

                if probs[row_idx, pos_idx] >= probs[row_idx, neg_idx]:
                    preds[row_idx, neg_idx] = 0
                else:
                    preds[row_idx, pos_idx] = 0

    return preds, conflict_count


def decode_joint_labels(binary_row, label_cols):
    decoded = []

    for value, label in zip(binary_row, label_cols):
        if int(value) == 1:
            aspect, sentiment = label.rsplit("_", 1)
            decoded.append(f"{aspect}:{sentiment}")

    return "; ".join(decoded)


def decode_aspects_only(binary_row, label_cols):
    aspects = []

    for value, label in zip(binary_row, label_cols):
        if int(value) == 1:
            aspect, _sentiment = label.rsplit("_", 1)
            aspects.append(aspect)

    return "; ".join(sorted(set(aspects)))


def get_top_candidate(probs_row, label_cols):
    idx = int(np.argmax(probs_row))
    label = label_cols[idx]
    aspect, sentiment = label.rsplit("_", 1)
    return label, aspect, sentiment, float(probs_row[idx])


def predict_joint_absa_batched(texts, batch_size=BATCH_SIZE):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc="Auto-labeling Shopee reviews"):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.sigmoid(outputs.logits.float()).detach().cpu().numpy()

        all_probs.append(probs)

    probs = np.vstack(all_probs)

    preds, conflict_count = apply_thresholds_with_conflict_resolution(
        probs=probs,
        thresholds=best_thresholds,
        label_cols=JOINT_LABEL_COLS,
        aspect_cols=ASPECT_COLS,
    )

    return probs, preds, conflict_count

In [8]:
# ============================================================
# 8. Run auto-labeling
# ============================================================

texts = df[TEXT_COL].tolist()

probs, preds, conflict_count = predict_joint_absa_batched(texts, batch_size=BATCH_SIZE)

print("Probability matrix shape:", probs.shape)
print("Prediction matrix shape :", preds.shape)
print("Conflicts resolved      :", conflict_count)

clean_gpu_memory()

Auto-labeling Shopee reviews:   0%|          | 0/1023 [00:00<?, ?it/s]

Probability matrix shape: (8179, 28)
Prediction matrix shape : (8179, 28)
Conflicts resolved      : 10


In [9]:
# ============================================================
# 9. Build auto-label outputs
# ============================================================

result_df = df.copy()

result_df["pred_aspect_sentiment"] = [
    decode_joint_labels(row, JOINT_LABEL_COLS)
    for row in preds
]

result_df["pred_aspects"] = [
    decode_aspects_only(row, JOINT_LABEL_COLS)
    for row in preds
]

result_df["pred_label_count"] = preds.sum(axis=1).astype(int)
result_df["pred_aspect_count"] = result_df["pred_aspects"].apply(
    lambda x: 0 if str(x).strip() == "" else len(str(x).split("; "))
)
result_df["no_aspect_predicted"] = result_df["pred_label_count"].eq(0)

top_candidates = [get_top_candidate(row, JOINT_LABEL_COLS) for row in probs]
result_df["top_candidate_label"] = [x[0] for x in top_candidates]
result_df["top_candidate_aspect"] = [x[1] for x in top_candidates]
result_df["top_candidate_sentiment"] = [x[2] for x in top_candidates]
result_df["top_candidate_probability"] = [x[3] for x in top_candidates]

# Add binary prediction columns.
for idx, label in enumerate(JOINT_LABEL_COLS):
    result_df[f"pred_{label}"] = preds[:, idx].astype(int)

# Add probability columns.
for idx, label in enumerate(JOINT_LABEL_COLS):
    result_df[f"prob_{label}"] = probs[:, idx].astype(float)

compact_cols = [
    ID_COL,
    TEXT_COL,
    "pred_aspects",
    "pred_aspect_sentiment",
    "pred_label_count",
    "pred_aspect_count",
    "no_aspect_predicted",
    "top_candidate_label",
    "top_candidate_probability",
]

compact_df = result_df[compact_cols].copy()

pass_only_df = compact_df[~compact_df["no_aspect_predicted"]].copy()
no_aspect_df = compact_df[compact_df["no_aspect_predicted"]].copy()

print("All rows             :", len(result_df))
print("Rows with prediction :", len(pass_only_df))
print("No-aspect rows       :", len(no_aspect_df))
print("No-aspect percentage :", round(len(no_aspect_df) / max(len(result_df), 1) * 100, 2), "%")

display(compact_df.head(10))
display(compact_df["pred_aspect_count"].value_counts().sort_index().to_frame("row_count"))

All rows             : 8179
Rows with prediction : 6030
No-aspect rows       : 2149
No-aspect percentage : 26.27 %


,id,english_review,pred_aspects,pred_aspect_sentiment,pred_label_count,pred_aspect_count,no_aspect_predicted,top_candidate_label,top_candidate_probability
0,2d68668cf8c3d9b8,i hate your ads,Trust,Trust:Negative,1,1,False,Trust_Negative,0.871529
1,d363df62267eae48,"Shopee please do not raise the service fee, from 1,000 then 2,000 now 2,200. If you could please go back to 1,000.",Trust,Trust:Negative,1,1,False,Trust_Negative,0.438804
2,8bdfe8eba767c20d,The most maximal apps have pop ads everywhere.,Item_quality,Item_quality:Negative,1,1,False,Trust_Negative,0.291796
3,c6e103e312de60a4,"Open the application, and then there's this live sale. It's not fun, especially with the full volume, set from the suit, just couldn't help. Please help me solve my problem.",,,0,0,True,Delivered_product_status_Negative,0.239723
4,c49004b4a142159c,"So satisfied and busy shopping in a diphoope, may the first-rate shooter",Trust,Trust:Positive,1,1,False,Trust_Positive,0.520214
5,52e7ed4683508031,"There's also broken stuff, and something doesn't match the order.",Delivered_product_status; Item_quality; Product_features,Product_features:Negative; Item_quality:Negative; Delivered_product_status:Negative,3,3,False,Delivered_product_status_Negative,0.590504
6,45cc42de4e2ba534,"Please don't do too much commercial"" or button passes zoomed in, I'm working on an apk editor show up an ad, and then I press through, and then I go through the apk shopee, and...",Packaging; Product_features,Product_features:Negative; Packaging:Negative,2,2,False,Product_features_Negative,0.297641
7,83cdbca62c7d8b77,"Too many ads. It's annoying, a little bit of an udh ad on the dial x. Keep it open.",Item_quality; Product_features; Trust,Product_features:Negative; Item_quality:Negative; Trust:Negative,3,3,False,Trust_Negative,0.580174
8,bb2df138ba17108d,My paansi is on any application if the shoope trs ad appears,,,0,0,True,Delivered_product_status_Negative,0.148154
9,cd12de26fc793564,A very helpful application of both easy and safe shopping,App_experience; Trust,App_experience:Positive; Trust:Positive,2,2,False,Trust_Positive,0.654198


,row_count
pred_aspect_count,
0,2149
1,3416
2,1411
3,537
4,352
5,179
6,81
7,37
8,11


In [ ]:
# ============================================================
# 10. Save auto-label outputs
# ============================================================

# Full file
result_df.to_csv(AUTO_FULL_PATH, index=False, encoding="utf-8-sig")

# Compact file: for quick inspection.
compact_df.to_csv(AUTO_COMPACT_PATH, index=False, encoding="utf-8-sig")

# Pass-only file
pass_only_df.to_csv(AUTO_PASS_ONLY_PATH, index=False, encoding="utf-8-sig")

# No-aspect audit
no_aspect_df.to_csv(NO_ASPECT_AUDIT_PATH, index=False, encoding="utf-8-sig")

print("Saved full probabilities :", AUTO_FULL_PATH)
print("Saved compact predictions:", AUTO_COMPACT_PATH)
print("Saved pass-only          :", AUTO_PASS_ONLY_PATH)
print("Saved no-aspect audit    :", NO_ASPECT_AUDIT_PATH)

Saved full probabilities : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect\06_shopee_autolabeled_deberta_14aspect_full_probabilities.csv
Saved compact predictions: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect\06_shopee_autolabeled_deberta_14aspect_compact.csv
Saved pass-only          : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect\06_shopee_autolabeled_deberta_14aspect_pass_only.csv
Saved no-aspect audit    : C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect\06_shopee_no_14aspect_predicted_audit.csv


In [ ]:
# ============================================================
# 11. Create human verification template
# ============================================================

human_template_df = compact_df.copy()

human_template_df["manual_action"] = ""
human_template_df["manual_corrected_aspect_sentiment"] = ""
human_template_df["manual_notes"] = ""

# Suggested action:
# - keep: prediction is correct
# - correct: aspect/sentiment needs correction
# - drop: review is spam, irrelevant, unclear, or unusable
# - no_aspect: review truly has no target aspect
# - unsure: needs second review

human_template_df.to_csv(
    HUMAN_VERIFICATION_TEMPLATE_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved human verification template:", HUMAN_VERIFICATION_TEMPLATE_PATH)
display(human_template_df.head(10))

Saved human verification template: C:\Users\Lenovo\Documents\Skripsi\dataset\phase2\shopee\autolabel_deberta_14aspect\07_shopee_human_verification_template.csv


,id,english_review,pred_aspects,pred_aspect_sentiment,pred_label_count,pred_aspect_count,no_aspect_predicted,top_candidate_label,top_candidate_probability,manual_action,manual_corrected_aspect_sentiment,manual_notes
0,2d68668cf8c3d9b8,i hate your ads,Trust,Trust:Negative,1,1,False,Trust_Negative,0.871529,,,
1,d363df62267eae48,"Shopee please do not raise the service fee, from 1,000 then 2,000 now 2,200. If you could please go back to 1,000.",Trust,Trust:Negative,1,1,False,Trust_Negative,0.438804,,,
2,8bdfe8eba767c20d,The most maximal apps have pop ads everywhere.,Item_quality,Item_quality:Negative,1,1,False,Trust_Negative,0.291796,,,
3,c6e103e312de60a4,"Open the application, and then there's this live sale. It's not fun, especially with the full volume, set from the suit, just couldn't help. Please help me solve my problem.",,,0,0,True,Delivered_product_status_Negative,0.239723,,,
4,c49004b4a142159c,"So satisfied and busy shopping in a diphoope, may the first-rate shooter",Trust,Trust:Positive,1,1,False,Trust_Positive,0.520214,,,
5,52e7ed4683508031,"There's also broken stuff, and something doesn't match the order.",Delivered_product_status; Item_quality; Product_features,Product_features:Negative; Item_quality:Negative; Delivered_product_status:Negative,3,3,False,Delivered_product_status_Negative,0.590504,,,
6,45cc42de4e2ba534,"Please don't do too much commercial"" or button passes zoomed in, I'm working on an apk editor show up an ad, and then I press through, and then I go through the apk shopee, and...",Packaging; Product_features,Product_features:Negative; Packaging:Negative,2,2,False,Product_features_Negative,0.297641,,,
7,83cdbca62c7d8b77,"Too many ads. It's annoying, a little bit of an udh ad on the dial x. Keep it open.",Item_quality; Product_features; Trust,Product_features:Negative; Item_quality:Negative; Trust:Negative,3,3,False,Trust_Negative,0.580174,,,
8,bb2df138ba17108d,My paansi is on any application if the shoope trs ad appears,,,0,0,True,Delivered_product_status_Negative,0.148154,,,
9,cd12de26fc793564,A very helpful application of both easy and safe shopping,App_experience; Trust,App_experience:Positive; Trust:Positive,2,2,False,Trust_Positive,0.654198,,,


In [12]:
# ============================================================
# 12. Save run summary
# ============================================================

summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "input_csv_path": str(INPUT_CSV_PATH),
    "best_model_dir": str(BEST_MODEL_DIR),
    "best_threshold_path": str(BEST_THRESHOLD_PATH),
    "selected_best_run_name_expected": SELECTED_BEST_RUN_NAME,
    "max_length": int(MAX_LENGTH),
    "batch_size": int(BATCH_SIZE),
    "device": str(DEVICE),
    "num_input_rows": int(len(df)),
    "num_output_rows": int(len(result_df)),
    "num_rows_with_at_least_one_prediction": int(len(pass_only_df)),
    "num_no_aspect_predicted_rows": int(len(no_aspect_df)),
    "no_aspect_predicted_percentage": float(len(no_aspect_df) / max(len(result_df), 1) * 100),
    "conflict_count_before_resolution": int(conflict_count),
    "aspect_schema_count": int(len(ASPECT_COLS)),
    "joint_label_count": int(len(JOINT_LABEL_COLS)),
    "aspect_schema": ASPECT_COLS,
    "joint_label_schema": JOINT_LABEL_COLS,
    "output_files": {
        "full_probabilities": str(AUTO_FULL_PATH),
        "compact_predictions": str(AUTO_COMPACT_PATH),
        "pass_only": str(AUTO_PASS_ONLY_PATH),
        "no_aspect_audit": str(NO_ASPECT_AUDIT_PATH),
        "human_verification_template": str(HUMAN_VERIFICATION_TEMPLATE_PATH),
    },
}

with open(RUN_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("Saved summary:", RUN_SUMMARY_PATH)

{
  "created_at_utc": "2026-06-20T15:38:35.226638+00:00",
  "input_csv_path": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase2\\shopee\\05_shopee_ready_for_deberta_autolabel_14aspect_no_aspect_filter.csv",
  "best_model_dir": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_sentiment_output_colab_base\\colab_best_joint_deberta_model",
  "best_threshold_path": "C:\\Users\\Lenovo\\Documents\\Skripsi\\dataset\\phase1\\trustpilot_reformatted_output\\deberta_joint_aspect_sentiment_output_colab_base\\colab_best_per_label_thresholds.json",
  "selected_best_run_name_expected": "colab_small_lr3e-5_len256_unweighted_epoch8",
  "max_length": 256,
  "batch_size": 8,
  "device": "cuda",
  "num_input_rows": 8179,
  "num_output_rows": 8179,
  "num_rows_with_at_least_one_prediction": 6030,
  "num_no_aspect_predicted_rows": 2149,
  "no_aspect_predicted_percentage": 26.274605697518034,
  "conflict_count_before_resolution": 10,
  "aspect_